# Lab 3 · Prompt engineering อย่างเป็นระบบ
**สัปดาห์ 3 · ช่วงบ่าย (50 นาที)**

เปลี่ยนการเขียน prompt จาก "ลองผิดลองถูก" เป็นการทดลองที่วัดผลได้
- โจทย์ **(ข)** สกัดชื่อสินค้าและราคาจากข้อความ → JSON
- โจทย์ **(ค)** จัดหมวดหมู่คำร้องเรียน 4 หมวด
- ชุด **dev 10 ข้อ** ใช้ปรับ prompt · ชุด **test 20 ข้อ** ห้ามเปิดจนถึง Lab 4
- ใช้ **greedy decoding** (`do_sample=False`) ตลอดแล็บนี้ เพื่อให้ผลซ้ำได้
- **เป้า** ของแล็บนี้: คะแนน dev ≥ 0.9 (dev มี 10 ข้อ → 0.1 = 1 ข้อ ผิดได้ไม่เกิน 1 ข้อ)

> **ก่อนรัน:** ปิด runtime ของ Lab 2 ก่อน (Runtime → Disconnect and delete runtime) ไม่งั้น GPU อาจเต็ม · Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install -q -U "transformers>=4.51,<5" accelerate
import torch, json, re, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings, transformers; warnings.filterwarnings("ignore"); transformers.utils.logging.set_verbosity_error()   # ซ่อน warning ที่ไม่เกี่ยวกับบทเรียน
MODEL = "scb10x/llama3.2-typhoon2-1b-instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto").eval()
EOT = tok.convert_tokens_to_ids("<|eot_id|>")      # token จบคำตอบ — generate หยุดเมื่อเจอตัวนี้
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่พบ GPU — เปลี่ยน runtime เป็น T4")
print("โหลดเสร็จ")

In [ ]:
#@title ชุดข้อมูลของสัปดาห์ 3 (ฝังไว้ในไฟล์ ไม่ต้องอัปโหลด · โค้ดถูกซ่อน ดับเบิลคลิกถ้าอยากดู) { display-mode: "form" }
# extract  = โจทย์ (ข) สกัดชื่อสินค้าและราคา  |  classify = โจทย์ (ค) จัดหมวดหมู่คำร้องเรียน 4 หมวด
# dev/test เขียนด้วยมือ · train 120 ข้อสร้างจากแม่แบบ (ข้อมูลสังเคราะห์ ใช้เพื่อการเรียนเท่านั้น)
import json
DATA = json.loads(r'''{"extract": {"dev": [{"text": "หูฟังไร้สาย SoundPro X2 ราคา 1,290 บาท ส่งฟรี", "product": "หูฟังไร้สาย SoundPro X2", "price": 1290}, {"text": "ซื้อกระติกน้ำเก็บความเย็น Hydro 750ml มาในราคา 590 บาท คุ้มมาก", "product": "กระติกน้ำเก็บความเย็น Hydro 750ml", "price": 590}, {"text": "โปรวันนี้! เมาส์ไร้สาย Logi M240 เหลือเพียง 450 บาท", "product": "เมาส์ไร้สาย Logi M240", "price": 450}, {"text": "แป้นพิมพ์ mechanical KeyMaster 87 ตัวนี้ 2,490 บาท มีของแถมเป็นสายถัก", "product": "แป้นพิมพ์ mechanical KeyMaster 87", "price": 2490}, {"text": "เก้าอี้สำนักงาน ErgoFlex ราคา ฿3,990 รับประกัน 2 ปี", "product": "เก้าอี้สำนักงาน ErgoFlex", "price": 3990}, {"text": "หม้อทอดไร้น้ำมัน AirCook 5L ลดเหลือ 1,790 บาท จาก 2,590", "product": "หม้อทอดไร้น้ำมัน AirCook 5L", "price": 1790}, {"text": "ขายกล้อง instant Snapy Mini ราคา 2,150 บาท พร้อมฟิล์ม 1 กล่อง", "product": "กล้อง instant Snapy Mini", "price": 2150}, {"text": "รองเท้าวิ่ง SwiftRun 3 ไซส์ 42 ราคา 2,890 บาท สภาพใหม่", "product": "รองเท้าวิ่ง SwiftRun 3", "price": 2890}, {"text": "โคมไฟตั้งโต๊ะ LumiDesk ปรับแสงได้ 3 ระดับ 690 บาทเท่านั้น", "product": "โคมไฟตั้งโต๊ะ LumiDesk", "price": 690}, {"text": "สายชาร์จ USB-C ยาว 2 เมตร ยี่ห้อ Volt ราคา 199 บาท", "product": "สายชาร์จ USB-C ยาว 2 เมตร ยี่ห้อ Volt", "price": 199}], "test": [{"text": "พัดลมตั้งพื้น CoolBreeze 16 นิ้ว ราคา 1,150 บาท ส่งถึงบ้าน", "product": "พัดลมตั้งพื้น CoolBreeze 16 นิ้ว", "price": 1150}, {"text": "ลำโพงบลูทูธ BoomBox Mini ตัวเล็กเสียงดี 890 บาท", "product": "ลำโพงบลูทูธ BoomBox Mini", "price": 890}, {"text": "นาฬิกาอัจฉริยะ FitTime S ราคาโปร 2,990 บาท จากปกติ 3,990", "product": "นาฬิกาอัจฉริยะ FitTime S", "price": 2990}, {"text": "เครื่องชงกาแฟแคปซูล Brewly One ราคา ฿4,590 แถมแคปซูล 20 ชิ้น", "product": "เครื่องชงกาแฟแคปซูล Brewly One", "price": 4590}, {"text": "กระเป๋าเป้ TrailPack 30L กันน้ำ ราคา 1,490 บาท", "product": "กระเป๋าเป้ TrailPack 30L", "price": 1490}, {"text": "แท็บเล็ต NotePad 10 รุ่น Wi-Fi 64GB ขาย 6,900 บาท", "product": "แท็บเล็ต NotePad 10 รุ่น Wi-Fi 64GB", "price": 6900}, {"text": "ที่นอนยางพารา DreamSoft 6 ฟุต ราคา 12,500 บาท ส่งฟรีทั่วไทย", "product": "ที่นอนยางพารา DreamSoft 6 ฟุต", "price": 12500}, {"text": "เครื่องดูดฝุ่นไร้สาย DustAway V3 เหลือ 3,290 บาท ถึงสิ้นเดือน", "product": "เครื่องดูดฝุ่นไร้สาย DustAway V3", "price": 3290}, {"text": "จอมอนิเตอร์ ViewMax 27 นิ้ว 144Hz ราคา 5,990 บาท", "product": "จอมอนิเตอร์ ViewMax 27 นิ้ว 144Hz", "price": 5990}, {"text": "หมวกกันน็อก SafeRide รุ่นเต็มใบ 1,850 บาท มีให้เลือก 3 สี", "product": "หมวกกันน็อก SafeRide รุ่นเต็มใบ", "price": 1850}, {"text": "ไมโครโฟน USB CastPro ราคา 1,690 บาท เหมาะกับสตรีมเกม", "product": "ไมโครโฟน USB CastPro", "price": 1690}, {"text": "เตารีดไอน้ำ SteamEase ราคาเพียง 790 บาท", "product": "เตารีดไอน้ำ SteamEase", "price": 790}, {"text": "จักรยานพับได้ FoldGo 20 นิ้ว ราคา 7,900 บาท พร้อมประกัน", "product": "จักรยานพับได้ FoldGo 20 นิ้ว", "price": 7900}, {"text": "เครื่องฟอกอากาศ PureAir 300 ราคา ฿3,490 กรอง PM2.5", "product": "เครื่องฟอกอากาศ PureAir 300", "price": 3490}, {"text": "กระทะเคลือบ NonStick Pro 28 ซม. ราคา 650 บาท", "product": "กระทะเคลือบ NonStick Pro 28 ซม.", "price": 650}, {"text": "ปากกาสไตลัส InkTip 2 ราคา 1,290 บาท ใช้กับแท็บเล็ตทุกรุ่น", "product": "ปากกาสไตลัส InkTip 2", "price": 1290}, {"text": "เสื้อกันฝน RainShield แบบพกพา 350 บาท", "product": "เสื้อกันฝน RainShield แบบพกพา", "price": 350}, {"text": "ตู้เย็นมินิ ChillBox 45L ราคา 3,990 บาท เหมาะกับหอพัก", "product": "ตู้เย็นมินิ ChillBox 45L", "price": 3990}, {"text": "แบตเตอรี่สำรอง PowerGo 20000mAh ราคา 990 บาท ชาร์จเร็ว", "product": "แบตเตอรี่สำรอง PowerGo 20000mAh", "price": 990}, {"text": "โต๊ะทำงานปรับระดับ LiftDesk 120 ซม. ราคา 8,900 บาท", "product": "โต๊ะทำงานปรับระดับ LiftDesk 120 ซม.", "price": 8900}], "train": [{"text": "หูฟังไร้สาย EchoBud ราคาปกติ 890 บาท", "product": "หูฟังไร้สาย EchoBud", "price": 890}, {"text": "ลำโพงพกพา MiniBeat ลดเหลือ 3,490 บาท วันนี้เท่านั้น", "product": "ลำโพงพกพา MiniBeat", "price": 3490}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 1,490 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 1490}, {"text": "คีย์บอร์ดไร้สาย TypeLite ราคาปกติ 2,490 บาท", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 2490}, {"text": "กระติกน้ำ ThermoCup 500ml ราคาปกติ 350 บาท", "product": "กระติกน้ำ ThermoCup 500ml", "price": 350}, {"text": "หม้อหุงข้าว RiceMate 1.8L ตัวนี้ 199 บาท มีของแถม", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 199}, {"text": "รีวิว เครื่องปั่น BlendGo ซื้อมา 990 บาท ใช้ดี", "product": "เครื่องปั่น BlendGo", "price": 990}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาเพียง 890 บาท ส่งฟรี", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 890}, {"text": "รีวิว พัดลมพกพา BreezePocket ซื้อมา 2,490 บาท ใช้ดี", "product": "พัดลมพกพา BreezePocket", "price": 2490}, {"text": "กระเป๋าเดินทาง RollEasy 24 นิ้ว ตัวนี้ 3,490 บาท มีของแถม", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 3490}, {"text": "รองเท้าผ้าใบ UrbanStep ลดเหลือ 1,690 บาท วันนี้เท่านั้น", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1690}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ลดเหลือ 890 บาท วันนี้เท่านั้น", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 890}, {"text": "สั่ง นาฬิกาข้อมือ TimeCore มาแล้ว 2,990 บาท คุ้มมาก", "product": "นาฬิกาข้อมือ TimeCore", "price": 2990}, {"text": "ขาย แว่นกันแดด SunVue ในราคา 199 บาท", "product": "แว่นกันแดด SunVue", "price": 199}, {"text": "หมวกแก๊ป CapOne ราคาปกติ 690 บาท", "product": "หมวกแก๊ป CapOne", "price": 690}, {"text": "โปรพิเศษ แท่นชาร์จไร้สาย ChargePad เหลือ 259 บาท", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 259}, {"text": "โปรพิเศษ สายชาร์จ Lightning FastLink เหลือ 199 บาท", "product": "สายชาร์จ Lightning FastLink", "price": 199}, {"text": "ที่วางโทรศัพท์ HoldUp ราคาปกติ 2,490 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 2490}, {"text": "สั่ง กล้องติดรถ DriveCam HD มาแล้ว 1,690 บาท คุ้มมาก", "product": "กล้องติดรถ DriveCam HD", "price": 1690}, {"text": "เครื่องวัดความดัน HealthCheck ราคาปกติ 1,690 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 1690}, {"text": "ไม้เซลฟี่ SnapStick ลดเหลือ 2,190 บาท วันนี้เท่านั้น", "product": "ไม้เซลฟี่ SnapStick", "price": 2190}, {"text": "ขาย เต็นท์ 2 คน CampLite ในราคา 1,490 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 1490}, {"text": "ถุงนอน WarmNest ลดเหลือ 259 บาท วันนี้เท่านั้น", "product": "ถุงนอน WarmNest", "price": 259}, {"text": "ไฟฉาย LED BrightBeam ราคาเพียง 2,490 บาท ส่งฟรี", "product": "ไฟฉาย LED BrightBeam", "price": 2490}, {"text": "สั่ง เขียงไม้ ChopWood มาแล้ว 990 บาท คุ้มมาก", "product": "เขียงไม้ ChopWood", "price": 990}, {"text": "โปรพิเศษ มีดเชฟ SharpEdge 8 นิ้ว เหลือ 5,900 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 5900}, {"text": "รีวิว ชุดเครื่องนอน SoftDream 6 ฟุต ซื้อมา 1,890 บาท ใช้ดี", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 1890}, {"text": "หมอนยางพารา NeckEase ราคาปกติ 1,690 บาท", "product": "หมอนยางพารา NeckEase", "price": 1690}, {"text": "รีวิว พรมเช็ดเท้า StepDry ซื้อมา 1,490 บาท ใช้ดี", "product": "พรมเช็ดเท้า StepDry", "price": 1490}, {"text": "สั่ง ตะกร้าผ้า FoldBin มาแล้ว 3,990 บาท คุ้มมาก", "product": "ตะกร้าผ้า FoldBin", "price": 3990}, {"text": "หูฟังไร้สาย EchoBud ราคาเพียง 3,990 บาท ส่งฟรี", "product": "หูฟังไร้สาย EchoBud", "price": 3990}, {"text": "ลำโพงพกพา MiniBeat ราคา 1,290 บาท", "product": "ลำโพงพกพา MiniBeat", "price": 1290}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 990 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 990}, {"text": "คีย์บอร์ดไร้สาย TypeLite ลดเหลือ 7,900 บาท วันนี้เท่านั้น", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 7900}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 1,290 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 1290}, {"text": "หม้อหุงข้าว RiceMate 1.8L ราคาปกติ 3,990 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 3990}, {"text": "เครื่องปั่น BlendGo ราคาเพียง 490 บาท ส่งฟรี", "product": "เครื่องปั่น BlendGo", "price": 490}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาปกติ 5,900 บาท", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 5900}, {"text": "โปรพิเศษ พัดลมพกพา BreezePocket เหลือ 990 บาท", "product": "พัดลมพกพา BreezePocket", "price": 990}, {"text": "ขาย กระเป๋าเดินทาง RollEasy 24 นิ้ว ในราคา 490 บาท", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 490}, {"text": "รองเท้าผ้าใบ UrbanStep ตัวนี้ 2,490 บาท มีของแถม", "product": "รองเท้าผ้าใบ UrbanStep", "price": 2490}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ฿350 มีประกัน", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 350}, {"text": "สั่ง นาฬิกาข้อมือ TimeCore มาแล้ว 350 บาท คุ้มมาก", "product": "นาฬิกาข้อมือ TimeCore", "price": 350}, {"text": "แว่นกันแดด SunVue ราคา 590 บาท", "product": "แว่นกันแดด SunVue", "price": 590}, {"text": "สั่ง หมวกแก๊ป CapOne มาแล้ว 1,190 บาท คุ้มมาก", "product": "หมวกแก๊ป CapOne", "price": 1190}, {"text": "ขาย แท่นชาร์จไร้สาย ChargePad ในราคา 1,890 บาท", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 1890}, {"text": "สายชาร์จ Lightning FastLink ราคาปกติ 259 บาท", "product": "สายชาร์จ Lightning FastLink", "price": 259}, {"text": "ที่วางโทรศัพท์ HoldUp ราคา 4,590 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 4590}, {"text": "กล้องติดรถ DriveCam HD ราคาปกติ 1,690 บาท", "product": "กล้องติดรถ DriveCam HD", "price": 1690}, {"text": "รีวิว เครื่องวัดความดัน HealthCheck ซื้อมา 1,290 บาท ใช้ดี", "product": "เครื่องวัดความดัน HealthCheck", "price": 1290}, {"text": "รีวิว ไม้เซลฟี่ SnapStick ซื้อมา 990 บาท ใช้ดี", "product": "ไม้เซลฟี่ SnapStick", "price": 990}, {"text": "เต็นท์ 2 คน CampLite ราคา 890 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 890}, {"text": "ถุงนอน WarmNest ราคา 1,190 บาท", "product": "ถุงนอน WarmNest", "price": 1190}, {"text": "ขาย ไฟฉาย LED BrightBeam ในราคา 350 บาท", "product": "ไฟฉาย LED BrightBeam", "price": 350}, {"text": "รีวิว เขียงไม้ ChopWood ซื้อมา 4,590 บาท ใช้ดี", "product": "เขียงไม้ ChopWood", "price": 4590}, {"text": "มีดเชฟ SharpEdge 8 นิ้ว ราคาเพียง 259 บาท ส่งฟรี", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 259}, {"text": "โปรพิเศษ ชุดเครื่องนอน SoftDream 6 ฟุต เหลือ 1,890 บาท", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 1890}, {"text": "โปรพิเศษ หมอนยางพารา NeckEase เหลือ 4,590 บาท", "product": "หมอนยางพารา NeckEase", "price": 4590}, {"text": "พรมเช็ดเท้า StepDry ราคา 590 บาท", "product": "พรมเช็ดเท้า StepDry", "price": 590}, {"text": "ตะกร้าผ้า FoldBin ฿1,290 มีประกัน", "product": "ตะกร้าผ้า FoldBin", "price": 1290}, {"text": "หูฟังไร้สาย EchoBud ลดเหลือ 1,490 บาท วันนี้เท่านั้น", "product": "หูฟังไร้สาย EchoBud", "price": 1490}, {"text": "สั่ง ลำโพงพกพา MiniBeat มาแล้ว 1,690 บาท คุ้มมาก", "product": "ลำโพงพกพา MiniBeat", "price": 1690}, {"text": "รีวิว เมาส์เกมมิ่ง ClickX ซื้อมา 2,190 บาท ใช้ดี", "product": "เมาส์เกมมิ่ง ClickX", "price": 2190}, {"text": "คีย์บอร์ดไร้สาย TypeLite ราคาปกติ 1,690 บาท", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 1690}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 7,900 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 7900}, {"text": "หม้อหุงข้าว RiceMate 1.8L ราคาปกติ 490 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 490}, {"text": "โปรพิเศษ เครื่องปั่น BlendGo เหลือ 2,990 บาท", "product": "เครื่องปั่น BlendGo", "price": 2990}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาเพียง 1,890 บาท ส่งฟรี", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 1890}, {"text": "สั่ง พัดลมพกพา BreezePocket มาแล้ว 1,190 บาท คุ้มมาก", "product": "พัดลมพกพา BreezePocket", "price": 1190}, {"text": "รีวิว กระเป๋าเดินทาง RollEasy 24 นิ้ว ซื้อมา 990 บาท ใช้ดี", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 990}, {"text": "รีวิว รองเท้าผ้าใบ UrbanStep ซื้อมา 1,190 บาท ใช้ดี", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1190}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ราคา 1,290 บาท", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 1290}, {"text": "นาฬิกาข้อมือ TimeCore ราคาปกติ 1,890 บาท", "product": "นาฬิกาข้อมือ TimeCore", "price": 1890}, {"text": "แว่นกันแดด SunVue ราคา 1,290 บาท", "product": "แว่นกันแดด SunVue", "price": 1290}, {"text": "หมวกแก๊ป CapOne ราคาปกติ 1,690 บาท", "product": "หมวกแก๊ป CapOne", "price": 1690}, {"text": "แท่นชาร์จไร้สาย ChargePad ลดเหลือ 3,990 บาท วันนี้เท่านั้น", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 3990}, {"text": "สายชาร์จ Lightning FastLink ฿259 มีประกัน", "product": "สายชาร์จ Lightning FastLink", "price": 259}, {"text": "ที่วางโทรศัพท์ HoldUp ฿2,190 มีประกัน", "product": "ที่วางโทรศัพท์ HoldUp", "price": 2190}, {"text": "กล้องติดรถ DriveCam HD ฿7,900 มีประกัน", "product": "กล้องติดรถ DriveCam HD", "price": 7900}, {"text": "โปรพิเศษ เครื่องวัดความดัน HealthCheck เหลือ 4,590 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 4590}, {"text": "ไม้เซลฟี่ SnapStick ราคา 2,490 บาท", "product": "ไม้เซลฟี่ SnapStick", "price": 2490}, {"text": "เต็นท์ 2 คน CampLite ราคา 3,990 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 3990}, {"text": "ถุงนอน WarmNest ราคา 7,900 บาท", "product": "ถุงนอน WarmNest", "price": 7900}, {"text": "โปรพิเศษ ไฟฉาย LED BrightBeam เหลือ 1,490 บาท", "product": "ไฟฉาย LED BrightBeam", "price": 1490}, {"text": "เขียงไม้ ChopWood ตัวนี้ 5,900 บาท มีของแถม", "product": "เขียงไม้ ChopWood", "price": 5900}, {"text": "มีดเชฟ SharpEdge 8 นิ้ว ราคาปกติ 1,190 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 1190}, {"text": "ชุดเครื่องนอน SoftDream 6 ฟุต ฿4,590 มีประกัน", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 4590}, {"text": "หมอนยางพารา NeckEase ฿690 มีประกัน", "product": "หมอนยางพารา NeckEase", "price": 690}, {"text": "พรมเช็ดเท้า StepDry ฿690 มีประกัน", "product": "พรมเช็ดเท้า StepDry", "price": 690}, {"text": "ตะกร้าผ้า FoldBin ราคาปกติ 1,490 บาท", "product": "ตะกร้าผ้า FoldBin", "price": 1490}, {"text": "โปรพิเศษ หูฟังไร้สาย EchoBud เหลือ 990 บาท", "product": "หูฟังไร้สาย EchoBud", "price": 990}, {"text": "ขาย ลำโพงพกพา MiniBeat ในราคา 1,690 บาท", "product": "ลำโพงพกพา MiniBeat", "price": 1690}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 199 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 199}, {"text": "คีย์บอร์ดไร้สาย TypeLite ลดเหลือ 7,900 บาท วันนี้เท่านั้น", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 7900}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 1,190 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 1190}, {"text": "โปรพิเศษ หม้อหุงข้าว RiceMate 1.8L เหลือ 890 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 890}, {"text": "เครื่องปั่น BlendGo ฿890 มีประกัน", "product": "เครื่องปั่น BlendGo", "price": 890}, {"text": "สั่ง โคมไฟอ่านหนังสือ ReadLite มาแล้ว 690 บาท คุ้มมาก", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 690}, {"text": "ขาย พัดลมพกพา BreezePocket ในราคา 5,900 บาท", "product": "พัดลมพกพา BreezePocket", "price": 5900}, {"text": "กระเป๋าเดินทาง RollEasy 24 นิ้ว ราคาปกติ 490 บาท", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 490}, {"text": "รองเท้าผ้าใบ UrbanStep ฿1,290 มีประกัน", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1290}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ราคาเพียง 7,900 บาท ส่งฟรี", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 7900}, {"text": "นาฬิกาข้อมือ TimeCore ลดเหลือ 2,190 บาท วันนี้เท่านั้น", "product": "นาฬิกาข้อมือ TimeCore", "price": 2190}, {"text": "แว่นกันแดด SunVue ฿350 มีประกัน", "product": "แว่นกันแดด SunVue", "price": 350}, {"text": "หมวกแก๊ป CapOne ราคาเพียง 5,900 บาท ส่งฟรี", "product": "หมวกแก๊ป CapOne", "price": 5900}, {"text": "แท่นชาร์จไร้สาย ChargePad ตัวนี้ 3,990 บาท มีของแถม", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 3990}, {"text": "สายชาร์จ Lightning FastLink ราคาเพียง 990 บาท ส่งฟรี", "product": "สายชาร์จ Lightning FastLink", "price": 990}, {"text": "ที่วางโทรศัพท์ HoldUp ราคา 490 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 490}, {"text": "กล้องติดรถ DriveCam HD ราคาเพียง 2,990 บาท ส่งฟรี", "product": "กล้องติดรถ DriveCam HD", "price": 2990}, {"text": "เครื่องวัดความดัน HealthCheck ราคาปกติ 1,290 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 1290}, {"text": "โปรพิเศษ ไม้เซลฟี่ SnapStick เหลือ 690 บาท", "product": "ไม้เซลฟี่ SnapStick", "price": 690}, {"text": "ขาย เต็นท์ 2 คน CampLite ในราคา 1,290 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 1290}, {"text": "ถุงนอน WarmNest ฿4,590 มีประกัน", "product": "ถุงนอน WarmNest", "price": 4590}, {"text": "ไฟฉาย LED BrightBeam ลดเหลือ 3,990 บาท วันนี้เท่านั้น", "product": "ไฟฉาย LED BrightBeam", "price": 3990}, {"text": "โปรพิเศษ เขียงไม้ ChopWood เหลือ 1,890 บาท", "product": "เขียงไม้ ChopWood", "price": 1890}, {"text": "โปรพิเศษ มีดเชฟ SharpEdge 8 นิ้ว เหลือ 2,990 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 2990}, {"text": "ชุดเครื่องนอน SoftDream 6 ฟุต ฿2,190 มีประกัน", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 2190}, {"text": "สั่ง หมอนยางพารา NeckEase มาแล้ว 5,900 บาท คุ้มมาก", "product": "หมอนยางพารา NeckEase", "price": 5900}, {"text": "สั่ง พรมเช็ดเท้า StepDry มาแล้ว 1,190 บาท คุ้มมาก", "product": "พรมเช็ดเท้า StepDry", "price": 1190}, {"text": "สั่ง ตะกร้าผ้า FoldBin มาแล้ว 3,990 บาท คุ้มมาก", "product": "ตะกร้าผ้า FoldBin", "price": 3990}]}, "classify": {"categories": ["การจัดส่ง", "สินค้าชำรุด", "การชำระเงิน", "บริการลูกค้า"], "dev": [{"text": "สั่งของมา 10 วันแล้วยังไม่ได้รับเลย สถานะขึ้นว่ากำลังจัดส่งตลอด", "label": "การจัดส่ง"}, {"text": "เปิดกล่องมาแล้วหน้าจอแตกเป็นรอยร้าว ใช้งานไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ตัดบัตรเครดิตไปแล้วสองครั้ง แต่ในระบบขึ้นว่ายังไม่ชำระ", "label": "การชำระเงิน"}, {"text": "โทรไปถามเรื่องสินค้า พนักงานพูดจาไม่ดีและวางสายใส่", "label": "บริการลูกค้า"}, {"text": "พัสดุมาถึงแต่กล่องบุบยับเยิน ของข้างในหักครึ่ง", "label": "สินค้าชำรุด"}, {"text": "แจ้งว่าจะส่งวันจันทร์ แต่เลื่อนมาเป็นศุกร์โดยไม่บอกล่วงหน้า", "label": "การจัดส่ง"}, {"text": "ขอคืนเงินไปสองสัปดาห์แล้ว ยังไม่ได้เงินคืนเข้าบัญชี", "label": "การชำระเงิน"}, {"text": "แชทถามไปตั้งแต่เช้า ตอบมาตอนสามทุ่มด้วยข้อความอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "ใช้โค้ดส่วนลดแล้วราคาไม่ลด แต่ระบบยังเรียกเก็บเต็มจำนวน", "label": "การชำระเงิน"}, {"text": "ได้ของผิดสี สั่งสีดำแต่ส่งสีขาวมา แถมมีรอยขีดข่วน", "label": "สินค้าชำรุด"}], "test": [{"text": "คนส่งของโยนกล่องไว้หน้าประตูแล้วไปโดยไม่กดกริ่ง ของหายไปเลย", "label": "การจัดส่ง"}, {"text": "หูฟังข้างซ้ายไม่มีเสียงตั้งแต่แกะกล่อง", "label": "สินค้าชำรุด"}, {"text": "ระบบหักเงินซ้ำสองรอบสำหรับออเดอร์เดียว", "label": "การชำระเงิน"}, {"text": "พนักงานหน้าร้านไม่ยอมรับเปลี่ยนสินค้าทั้งที่ยังอยู่ในระยะประกัน และพูดจาไม่สุภาพ", "label": "บริการลูกค้า"}, {"text": "เลขพัสดุที่ให้มาตรวจสอบไม่ได้ ขึ้นว่าไม่พบข้อมูล", "label": "การจัดส่ง"}, {"text": "เครื่องปั่นใช้ได้สองครั้งแล้วมอเตอร์ไหม้ มีกลิ่นพลาสติก", "label": "สินค้าชำรุด"}, {"text": "โอนเงินผ่านแอปธนาคารแล้ว แต่ออเดอร์ถูกยกเลิกอัตโนมัติว่าไม่ชำระ", "label": "การชำระเงิน"}, {"text": "ติดต่อคอลเซ็นเตอร์รอสายนาน 40 นาที แล้วสายหลุด", "label": "บริการลูกค้า"}, {"text": "สินค้ามาถึงช้ากว่ากำหนดหนึ่งสัปดาห์และไม่มีการแจ้งเตือนใด ๆ", "label": "การจัดส่ง"}, {"text": "โต๊ะที่ได้รับขาไม่เท่ากัน โยกไปมา ใช้งานไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ใบเสร็จระบุยอดไม่ตรงกับที่ถูกตัดบัตร ต่างกัน 300 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งปัญหาไปสามครั้ง ได้แต่คำตอบว่าจะติดต่อกลับ แต่ไม่เคยติดต่อมา", "label": "บริการลูกค้า"}, {"text": "ส่งมาผิดที่อยู่ ทั้งที่กรอกที่อยู่ถูกต้องตอนสั่ง", "label": "การจัดส่ง"}, {"text": "รองเท้าที่ได้รับพื้นหลุดออกจากตัวรองเท้าตั้งแต่วันแรก", "label": "สินค้าชำรุด"}, {"text": "เลือกผ่อน 0% แต่ระบบเรียกเก็บเต็มจำนวนในงวดแรก", "label": "การชำระเงิน"}, {"text": "พนักงานแชทตอบไม่ตรงคำถาม ถามเรื่องประกันแต่ตอบเรื่องโปรโมชัน", "label": "บริการลูกค้า"}, {"text": "พัสดุถูกส่งกลับไปที่ผู้ขายโดยที่ไม่มีใครมาส่งเลยสักครั้ง", "label": "การจัดส่ง"}, {"text": "ซองบรรจุแตก เมล็ดกาแฟหกออกมาเกือบครึ่งถุง", "label": "สินค้าชำรุด"}, {"text": "ยกเลิกออเดอร์ตั้งแต่ก่อนส่ง แต่เงินยังไม่ถูกคืนหลังผ่านไป 20 วัน", "label": "การชำระเงิน"}, {"text": "ขอใบกำกับภาษีไปแล้วสองสัปดาห์ ยังไม่มีใครดำเนินการให้", "label": "บริการลูกค้า"}], "train": [{"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 10 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "กระเป๋ามีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "แต้มสะสมถูกหักไปแล้ว แต่ส่วนลดไม่แสดงในยอดชำระ", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 10 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ระบบให้เลือกวันส่ง แต่ของมาก่อนวันที่เลือกแล้วไม่มีคนรับ", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำที่ส่งมาเป็นรุ่นเก่า ไม่ตรงกับที่โฆษณาไว้", "label": "สินค้าชำรุด"}, {"text": "เปลี่ยนวิธีจ่ายเป็นเก็บเงินปลายทาง แต่ระบบยังตัดบัตรอยู่ดี", "label": "การชำระเงิน"}, {"text": "ขอเอกสารรับประกันไปแล้ว 3 ครั้ง ยังไม่ได้รับ", "label": "บริการลูกค้า"}, {"text": "เปลี่ยนที่อยู่จัดส่งในระบบแล้ว แต่ยังส่งไปที่อยู่เก่า", "label": "การจัดส่ง"}, {"text": "เก้าอี้ใช้ไปแค่ 3 วัน ปุ่มก็กดไม่ติดแล้ว", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 300 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 7 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "กล่องใหญ่ถูกยัดใส่ตู้รับพัสดุจนต้องดึงออกไม่ได้", "label": "การจัดส่ง"}, {"text": "ขนาดของหูฟังเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 200 บาท", "label": "การชำระเงิน"}, {"text": "ช่องทางติดต่อบนเว็บเป็นเบอร์ที่โทรไม่ติดทุกครั้ง", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 7 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขนาดของลำโพงเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "สั่งซื้อด้วยคูปองเงินสด แต่ยอดคูปองหายไปและไม่ได้สินค้า", "label": "การชำระเงิน"}, {"text": "แชทบอทวนตอบเรื่องเดิม ไม่มีทางต่อสายกับคนจริง", "label": "บริการลูกค้า"}, {"text": "แอปแจ้งว่าส่งแล้ว แต่รูปหลักฐานเป็นหน้าบ้านคนอื่น", "label": "การจัดส่ง"}, {"text": "สีของพัดลมหลุดลอกหลังจากล้างครั้งเดียว", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 200 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "พนักงานให้ข้อมูลโปรโมชันผิด ทำให้เสียสิทธิ์ส่วนลด", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันศุกร์ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ซีลของสายชาร์จถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 5 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "ถามเรื่องสเปกสินค้า พนักงานบอกให้ไปหาเองในอินเทอร์เน็ต", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 2 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "ขนาดของกระจกเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 200 บาท", "label": "การชำระเงิน"}, {"text": "ขอคุยกับหัวหน้าแล้วถูกบอกว่าไม่มีใครรับผิดชอบเรื่องนี้", "label": "บริการลูกค้า"}, {"text": "จ่ายค่าส่งด่วนแล้ว แต่ของมาถึงช้ากว่าแบบธรรมดา", "label": "การจัดส่ง"}, {"text": "ซีลของลำโพงถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 500 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานหน้าร้านมองข้ามลูกค้าที่ยืนรอ แล้วไปคุยโทรศัพท์ส่วนตัว", "label": "บริการลูกค้า"}, {"text": "ของสดถูกวางตากแดดไว้หน้าบ้านทั้งวันกว่าจะมีคนบอก", "label": "การจัดส่ง"}, {"text": "กระเป๋าทำงานได้แค่ครึ่งเดียว อีกครึ่งไม่ตอบสนอง", "label": "สินค้าชำรุด"}, {"text": "ระบบเก็บค่าธรรมเนียมเพิ่มโดยไม่แจ้งก่อนกดชำระ", "label": "การชำระเงิน"}, {"text": "พนักงานปิดเคสให้เองทั้งที่ปัญหายังไม่ถูกแก้", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันพุธ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ขนาดของสายชาร์จเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "สมัครสมาชิกรายเดือนแล้ว แต่ถูกเรียกเก็บเป็นรายปี", "label": "การชำระเงิน"}, {"text": "ร้องเรียนเรื่องพนักงานแล้วถูกบล็อกจากเพจของร้าน", "label": "บริการลูกค้า"}, {"text": "แยกส่งเป็นสองกล่อง อีกกล่องหายไปเฉย ๆ ไม่มีใครตามให้", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ระบบแจ้งว่าบัตรถูกปฏิเสธ แต่ธนาคารยืนยันว่าตัดเงินไปแล้ว", "label": "การชำระเงิน"}, {"text": "พนักงานตอบแชทด้วยอีโมจิอย่างเดียว ไม่ตอบคำถามที่ถาม", "label": "บริการลูกค้า"}, {"text": "ขนส่งโทรมาแล้ววางสายทันที ไม่รอให้รับ แล้วตีกลับว่าติดต่อไม่ได้", "label": "การจัดส่ง"}, {"text": "เปิดกล่องมาลำโพงเปียกชื้น มีคราบน้ำอยู่ข้างใน", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 500 บาท", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 3 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 4 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำมีรอยแตกร้าวมาตั้งแต่ในกล่อง ดูเหมือนโดนกระแทกระหว่างทาง", "label": "สินค้าชำรุด"}, {"text": "ผ่อนชำระเดือนแรกถูกตัดสองงวดพร้อมกัน", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 3 วัน", "label": "บริการลูกค้า"}, {"text": "ขนส่งบอกว่าที่อยู่ไม่ชัดเจน ทั้งที่เคยส่งที่เดิมมาแล้วหลายครั้ง", "label": "การจัดส่ง"}, {"text": "ซีลของตัวเครื่องถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 30 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 10 วัน", "label": "บริการลูกค้า"}, {"text": "กรอกที่อยู่คอนโดพร้อมเลขห้อง แต่ของถูกฝากไว้ที่ร้านสะดวกซื้ออีกซอย", "label": "การจัดส่ง"}, {"text": "หลังชาร์จหน้าจอครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 3 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 2 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 5 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "หลังชาร์จสายชาร์จครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "โปรผ่อน 0% หายไปตอนกดชำระ กลายเป็นคิดดอกเบี้ยเต็ม", "label": "การชำระเงิน"}, {"text": "นัดช่างมาซ่อมที่บ้านแล้วไม่มา และไม่แจ้งยกเลิก", "label": "บริการลูกค้า"}, {"text": "ไรเดอร์เอาของไปส่งผิดบ้าน ต้องเดินไปขอคืนจากเพื่อนบ้านเอง", "label": "การจัดส่ง"}, {"text": "ลำโพงมีรอยแตกร้าวมาตั้งแต่ในกล่อง ดูเหมือนโดนกระแทกระหว่างทาง", "label": "สินค้าชำรุด"}, {"text": "เงินมัดจำถูกหักแล้ว แต่ออเดอร์ไม่ถูกสร้างในระบบ", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 2 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 14 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขาโต๊ะมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 300 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 7 วัน", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 10 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "หลังชาร์จกระจกครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ขอเปลี่ยนใบเสร็จเป็นชื่อบริษัทแล้วถูกเรียกเก็บเงินเพิ่ม", "label": "การชำระเงิน"}, {"text": "พนักงานพูดจาประชดเมื่อถามเรื่องการคืนสินค้า", "label": "บริการลูกค้า"}, {"text": "พนักงานส่งเรียกเก็บเงินปลายทางซ้ำ ทั้งที่จ่ายออนไลน์แล้ว", "label": "การจัดส่ง"}, {"text": "ลำโพงมีเสียงดังผิดปกติตอนเปิดใช้ครั้งแรก", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 100 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 20 วัน", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 5 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขาโต๊ะมีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่าน QR แล้วเงินออกจากบัญชี แต่หน้าเว็บขึ้นว่าชำระไม่สำเร็จ", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 30 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 3 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "หูฟังมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 500 บาท", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 4 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 3 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "กระจกขึ้นสนิมภายในสัปดาห์แรก", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 200 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 30 วัน", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันจันทร์ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ขนาดของพัดลมเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 20 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 4 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 3 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "เก้าอี้ประกอบไม่ได้เพราะรูสกรูไม่ตรงกัน", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 100 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 5 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 30 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ลำโพงมีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 300 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 3 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 20 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "พัดลมทำงานได้แค่ครึ่งเดียว อีกครึ่งไม่ตอบสนอง", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 100 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 3 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 20 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "สายชาร์จมีเสียงดังผิดปกติตอนเปิดใช้ครั้งแรก", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 300 บาท", "label": "การชำระเงิน"}, {"text": "ขอเอกสารรับประกันไปแล้ว 2 ครั้ง ยังไม่ได้รับ", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 30 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "เก้าอี้ที่ส่งมาเป็นรุ่นเก่า ไม่ตรงกับที่โฆษณาไว้", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 100 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 5 วัน", "label": "บริการลูกค้า"}]}}''')
for task in ("extract", "classify"):
    print(task, {k: len(v) for k, v in DATA[task].items() if isinstance(v, list)})
print("ตัวอย่าง extract/dev:", DATA["extract"]["dev"][0])
print("ตัวอย่าง classify/dev:", DATA["classify"]["dev"][0])

## 3.1 ฟังก์ชันเรียกโมเดล และฟังก์ชันวัดผล
งานต่างชนิดวัดคนละแบบ แต่ทั้งสองแบบ **นับ "รูปแบบ" เป็นส่วนหนึ่งของความถูกต้อง** เพราะระบบจริงต้องเอาคำตอบไปใช้ต่อโดยอัตโนมัติ

| โจทย์ | คะแนน (0–1) | "ตอบตรงรูปแบบ" |
|---|---|---|
| (ข) สกัด | ได้ JSON + `product` ตรง 0.5 (ตรงบางส่วน 0.25) + `price` ตรง 0.5 | **ทั้งคำตอบ** เป็น JSON ที่มี `product` และ `price` (ห่อด้วย ``` ได้ แต่ห้ามมีคำอธิบายปน) |
| (ค) จัดหมวด | **บรรทัดแรก** ของคำตอบเป็นชื่อหมวดที่ถูกต้องพอดี → 1 ไม่งั้น 0 | บรรทัดแรกเป็นชื่อหมวดใดหมวดหนึ่ง |

ทำไมเข้มแบบนี้: ลองนึกว่าคำตอบถูกส่งต่อให้โปรแกรมอื่นอ่านเอง — โปรแกรมอ่าน "ของมาช้าค่ะ น่าจะเป็นหมวดการจัดส่งนะคะ" ไม่ออก
- (ค) ตอบยาวเป็นย่อหน้าแม้จะพูดถึงหมวดที่ถูก = 0 คะแนน (แต่นับแยกไว้ในคอลัมน์ "ถูกหมวดแต่ตอบยาว" ให้เห็นว่าโมเดลรู้คำตอบ แค่ไม่ทำตามรูปแบบ)
- (ข) ถ้ามีคำอธิบายปนกับ JSON ยังได้คะแนนเนื้อหา (เพราะหยิบก้อน `{...}` ออกมาตรวจ) แต่ไม่นับว่า "ตอบตรงรูปแบบ"

ฟังก์ชันวัดผลชุดนี้ใช้ใน Lab 4 ด้วย **ทุกตัวอักษร** — คะแนนสองแล็บจึงเทียบกันได้

In [ ]:
@torch.no_grad()
def ask(system, user, max_new_tokens=80):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    do_sample = False     # greedy → ผลซ้ำได้
    g = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens, do_sample=do_sample,
                       eos_token_id=EOT, pad_token_id=EOT)
    return tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip(), ids.shape[1]

def strip_wrap(s):
    # ตัดคำนำหน้า "คำตอบ:" และ ``` ที่ห่อ JSON ออก (ยอมให้มีได้ ไม่นับว่าผิดรูปแบบ)
    s = re.sub(r"^(คำตอบ|answer)\s*[:：]\s*", "", s.strip(), flags=re.I)
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", s).strip()

def parse_json_obj(s):
    # ใช้ให้คะแนน "เนื้อหา": ลอง parse ทั้งคำตอบก่อน ถ้ามีข้อความปนค่อยหยิบก้อน {...} ก้อนแรก
    try:
        o = json.loads(strip_wrap(s))
        if isinstance(o, dict): return o
    except Exception: pass
    m = re.search(r"\{.*?\}", s, re.S)
    if not m: return None
    try:
        o = json.loads(m.group(0)); return o if isinstance(o, dict) else None
    except Exception: return None

def norm(s): return re.sub(r"\s+", " ", str(s)).strip()
def score_extract(output, item):
    # product ตรงเป๊ะ 0.5 · ตรงบางส่วน (ชื่อสั้น/ยาวกว่าเฉลย) 0.25 · price ตรง 0.5
    obj = parse_json_obj(output)
    if obj is None: return 0.0
    s = 0.0
    p_out, p_gold = norm(obj.get("product", "")), norm(item["product"])
    if p_out == p_gold: s += 0.5
    elif p_out and (p_out in p_gold or p_gold in p_out): s += 0.25
    try:
        if int(float(str(obj.get("price", "")).replace(",", "").replace("฿", ""))) == item["price"]: s += 0.5
    except Exception: pass
    return s

CATS = DATA["classify"]["categories"]
def first_answer(output):
    # บรรทัดแรกของคำตอบ ตัดคำนำหน้า "หมวด:" / "คำตอบ:" และเครื่องหมายหัวท้ายออก
    line = output.strip().split("\n")[0]
    line = re.sub(r"^(หมวด|คำตอบ)\s*[:：]\s*", "", line.strip())
    return line.strip(" .!\"'`*")
def score_classify(output, item):
    return 1.0 if first_answer(output) == item["label"] else 0.0     # บรรทัดแรกต้องเป็นชื่อหมวดที่ถูก "พอดี"

def format_ok(task, output):
    # "ตอบตรงรูปแบบ" = ระบบปลายทางเอาไปใช้ได้ทันทีโดยไม่ต้องตัดต่อ
    if task == "classify": return first_answer(output) in CATS      # บรรทัดแรกเป็นชื่อหมวดใดหมวดหนึ่ง
    try: obj = json.loads(strip_wrap(output))                         # ทั้งคำตอบต้องเป็น JSON (ห้ามมีคำอธิบายปน)
    except Exception: return False
    return isinstance(obj, dict) and "product" in obj and "price" in obj
def mentions_gold_only(output, item):
    return [c for c in CATS if c in output] == [item["label"]]        # พูดถึงหมวดที่ถูกเพียงหมวดเดียว (ไม่สนรูปแบบ)

def evaluate(task, system, user_fn, split="dev", verbose=False):
    """คืน dict: คะแนนเฉลี่ย · สัดส่วนที่ตอบตรงรูปแบบ · (ค) จำนวนที่ถูกหมวดแต่ตอบยาว · token ของ prompt เฉลี่ย"""
    items = DATA[task][split]; scorer = score_extract if task == "extract" else score_classify
    total, fmt, long_ok, ntok = 0.0, 0, 0, 0
    for it in items:
        out, n = ask(system, user_fn(it["text"])); sc = scorer(out, it)
        total += sc; fmt += format_ok(task, out); ntok += n
        if task == "classify" and sc == 0 and mentions_gold_only(out, it): long_ok += 1
        if verbose: print(f"[{sc:.1f}] {it['text'][:40]}... → {out[:70]!r}")
    r = {"คะแนน dev": round(total / len(items), 2), "ตอบตรงรูปแบบ": f"{fmt}/{len(items)}", "token ของ prompt (เฉลี่ย)": round(ntok / len(items))}
    r["ถูกหมวดแต่ตอบยาว"] = long_ok if task == "classify" else "–"     # (ข) ไม่มีแนวคิดนี้
    return r

## 3.2 ไล่ระดับ prompt ทีละตัวแปร
เปลี่ยน **ทีละอย่าง** เพื่อให้รู้ว่าผลมาจากอะไร: L0 zero-shot → L1 + รูปแบบ output → L2 + few-shot 2 ตัวอย่าง → L3 + บทบาท
(ตัวอย่าง few-shot ไม่ซ้ำกับ dev/test)

In [ ]:
# ---------- โจทย์ (ข) สกัดข้อมูล ----------
EX_B = [('นาฬิกาปลุก WakeUp Mini ราคา 350 บาท', '{"product": "นาฬิกาปลุก WakeUp Mini", "price": 350}'),
        ('ขายพรมห้องนั่งเล่น CozyRug 2x3 เมตร ในราคา 2,190 บาท ส่งฟรี', '{"product": "พรมห้องนั่งเล่น CozyRug 2x3 เมตร", "price": 2190}')]
FEW_B = "\n".join(f'ข้อความ: {t}\nคำตอบ: {a}' for t, a in EX_B)

LEVELS_B = {
 "L0 zero-shot":        (None, lambda x: f"สกัดชื่อสินค้าและราคาจากข้อความนี้: {x}"),
 "L1 + รูปแบบ output":  (None, lambda x: f'สกัดชื่อสินค้าและราคาจากข้อความ ตอบเป็น JSON เท่านั้น รูปแบบ {{"product": ..., "price": <ตัวเลขไม่มีเครื่องหมาย>}}\nข้อความ: {x}\nคำตอบ:'),
 "L2 + few-shot":       (None, lambda x: f'สกัดชื่อสินค้าและราคาจากข้อความ ตอบเป็น JSON เท่านั้น รูปแบบ {{"product": ..., "price": <ตัวเลขไม่มีเครื่องหมาย>}}\n{FEW_B}\nข้อความ: {x}\nคำตอบ:'),
 "L3 + บทบาท":          ("คุณคือระบบสกัดข้อมูลสินค้าจากประกาศขายภาษาไทย",     # เพิ่มเฉพาะบทบาท ส่วน user เหมือน L2 ทุกตัวอักษร
                         lambda x: f'สกัดชื่อสินค้าและราคาจากข้อความ ตอบเป็น JSON เท่านั้น รูปแบบ {{"product": ..., "price": <ตัวเลขไม่มีเครื่องหมาย>}}\n{FEW_B}\nข้อความ: {x}\nคำตอบ:'),
}
# ---------- โจทย์ (ค) จัดหมวดหมู่ ----------
CAT_STR = " / ".join(CATS)
EX_C = [("ของมาถึงช้ากว่ากำหนดสามวัน", "การจัดส่ง"), ("แก้วที่ได้รับมีรอยร้าว", "สินค้าชำรุด")]
FEW_C = "\n".join(f"คำร้องเรียน: {t}\nหมวด: {a}" for t, a in EX_C)
LEVELS_C = {
 "L0 zero-shot":        (None, lambda x: f"คำร้องเรียนนี้อยู่ในหมวดไหน: {x}"),
 "L1 + รูปแบบ output":  (None, lambda x: f"จัดหมวดหมู่คำร้องเรียนต่อไปนี้ เลือกหนึ่งหมวดจาก [{CAT_STR}] ตอบเฉพาะชื่อหมวด\nคำร้องเรียน: {x}\nหมวด:"),
 "L2 + few-shot":       (None, lambda x: f"จัดหมวดหมู่คำร้องเรียนต่อไปนี้ เลือกหนึ่งหมวดจาก [{CAT_STR}] ตอบเฉพาะชื่อหมวด\n{FEW_C}\nคำร้องเรียน: {x}\nหมวด:"),
 "L3 + บทบาท":          ("คุณคือเจ้าหน้าที่คัดแยกคำร้องเรียนของร้านค้าออนไลน์",   # เพิ่มเฉพาะบทบาท ส่วน user เหมือน L2 ทุกตัวอักษร
                         lambda x: f"จัดหมวดหมู่คำร้องเรียนต่อไปนี้ เลือกหนึ่งหมวดจาก [{CAT_STR}] ตอบเฉพาะชื่อหมวด\n{FEW_C}\nคำร้องเรียน: {x}\nหมวด:"),
}

In [ ]:
# รัน 2 โจทย์ × 4 ระดับ × dev 10 ข้อ = 80 ครั้ง (ประมาณ 3–5 นาทีบน T4)
results = []
for task, levels in [("extract", LEVELS_B), ("classify", LEVELS_C)]:
    for name, (system, fn) in levels.items():
        r = evaluate(task, system, fn, "dev")
        results.append({"โจทย์": task, "ระดับ prompt": name, **r})
        print(f"เสร็จ: {task:9s} {name:22s} คะแนน {r['คะแนน dev']:.2f}")     # แสดงความคืบหน้าระหว่างรอ (ตารางเต็มอยู่ด้านล่าง)
df = pd.DataFrame(results)
df

**อ่านตารางอย่างไร** — ไล่ L0 → L3 ในแต่ละโจทย์ (dev 10 ข้อ: คะแนนต่างกัน 0.1 = ต่างกันแค่ 1 ข้อ อย่าตีความมากเกินไป)
- **คะแนน dev** ควรเพิ่มขึ้นเมื่อเพิ่ม "รูปแบบ output" (L1) และ "few-shot" (L2) · "บทบาท" (L3) มักช่วยน้อยกับโจทย์แบบนี้
- **ตอบตรงรูปแบบ** ของ L0 มักต่ำ — โมเดลตอบเป็นประโยค ไม่ใช่ JSON / ชื่อหมวด · คอลัมน์ **ถูกหมวดแต่ตอบยาว** บอกว่าโมเดล "รู้" คำตอบแต่ไม่ทำตามรูปแบบ → นี่คือสิ่งที่ prompt ระดับ L1 แก้
- **token ของ prompt** = ต้นทุนที่จ่ายเพิ่มในทุกครั้งที่เรียก — ถ้า L1 ได้คะแนนใกล้ L2 แปลว่าไม่ต้องจ่าย few-shot ก็ได้

> ดูตัวอย่างคำตอบจริงของระดับที่คะแนนต่ำ เพื่อเข้าใจว่าโมเดล "พลาด" แบบไหน (ตอบยาวเกิน? ตอบหลายหมวด? JSON ผิดรูป?)

In [ ]:
print("คำตอบจริงของ classify · L0 zero-shot (ตัวเลขในวงเล็บ = คะแนนข้อนั้น)")
r = evaluate("classify", *LEVELS_C["L0 zero-shot"], "dev", verbose=True)
print("สรุป:", r)

## 3.3 (แถวแยก) chain-of-thought กับโจทย์ (ค)
ให้โมเดล "คิดก่อนตอบ" — ทำเฉพาะโจทย์ (ค) เพราะโจทย์ (ข) ต้องตอบ JSON เท่านั้น · สังเกตว่าโมเดล 1B ได้ประโยชน์จาก CoT น้อยกว่าโมเดลใหญ่ และใช้ token ตอบมากขึ้น

In [ ]:
EX_C_COT = [("ของมาถึงช้ากว่ากำหนดสามวัน", "ปัญหาอยู่ที่ความล่าช้าของการส่ง", "การจัดส่ง"),
            ("แก้วที่ได้รับมีรอยร้าว", "ตัวสินค้าเสียหาย", "สินค้าชำรุด")]
FEW_C_COT = "\n".join(f"คำร้องเรียน: {t}\nเหตุผล: {r}\nหมวด: {a}" for t, r, a in EX_C_COT)   # ตัวอย่างต้องมีเหตุผลด้วย ไม่งั้นโมเดลจะเลียนแบบแล้วข้ามการอธิบาย

def cot_user(x):
    return (f"จัดหมวดหมู่คำร้องเรียนต่อไปนี้ เลือกหนึ่งหมวดจาก [{CAT_STR}]\n"
            f"ให้เขียนบรรทัด 'เหตุผล: <1 ประโยค>' ก่อน แล้วปิดท้ายด้วยบรรทัด 'หมวด: <ชื่อหมวด>'\n{FEW_C_COT}\nคำร้องเรียน: {x}")

def score_classify_cot(output, item):
    # ใช้บรรทัด "หมวด: ..." บรรทัดแรกของคำตอบ แล้วตัดสินแบบเข้มเท่ากับ score_classify (ต้องเป็นชื่อหมวดพอดี)
    m = re.search(r"หมวด\s*[:：]\s*(.+)", output)
    return 1.0 if m and first_answer(m.group(1)) == item["label"] else 0.0

dev = DATA["classify"]["dev"]
system_L2, fn_L2 = LEVELS_C["L2 + few-shot"]
tot_cot, tot_l2, len_cot, len_l2 = 0.0, 0.0, 0, 0
for i, it in enumerate(dev):
    out, _ = ask(None, cot_user(it["text"]), max_new_tokens=200); sc = score_classify_cot(out, it)
    out2, _ = ask(system_L2, fn_L2(it["text"]))
    tot_cot += sc; tot_l2 += score_classify(out2, it)
    len_cot += len(tok(out, add_special_tokens=False).input_ids); len_l2 += len(tok(out2, add_special_tokens=False).input_ids)          # ความยาวคำตอบ (token)
    if i < 2: print(f"ตัวอย่างคำตอบ CoT ข้อ {i+1} (เฉลย {it['label']}, คะแนน {sc:.0f}):\n{out}\n")
n = len(dev)
print(f"classify · L2 + few-shot (ไม่มี CoT): คะแนน {tot_l2/n:.2f} · คำตอบยาวเฉลี่ย {len_l2/n:.0f} token")
print(f"classify · L2 + CoT               : คะแนน {tot_cot/n:.2f} · คำตอบยาวเฉลี่ย {len_cot/n:.0f} token   ← CoT ช่วยไหม? คุ้มกับ token ที่เพิ่มไหม?")

## 3.4 เทียบกับ ChatGPT / Claude (ทำผ่านหน้าเว็บ ไม่ต้องใช้ API)
notebook เรียก ChatGPT/Claude ไม่ได้ จึงต้อง **คัดลอก prompt ไปวางในหน้าเว็บด้วยตัวเอง** — เซลล์ด้านล่างจะ

1. เลือก **1 โจทย์ × 3 ข้อ** จากชุด dev และ **2 ระดับ** (L0 กับ L3) = prompt 6 ชุด
2. ให้ Typhoon2 ตอบทั้ง 6 ชุดก่อน (คอลัมน์แรกของตารางจึงมีให้แล้ว)
3. พิมพ์ prompt แต่ละชุดระหว่างเส้นประ พร้อมเฉลย ให้คัดลอก **เฉพาะข้อความระหว่างเส้นประทั้งก้อน** ไปวางใน ChatGPT และ Claude แล้วจดว่าตอบถูกไหม

**กติกา:** เปิดแชทใหม่ทุกครั้ง ปิด memory / custom instructions · บรรทัดบทบาท (ถ้ามี) ถูกพิมพ์เป็นบรรทัดแรกของก้อนให้แล้ว เพราะหน้าเว็บไม่มีช่อง system · ปิดโหมด reasoning ถ้าปิดได้

In [ ]:
task, levels = "classify", LEVELS_C          # เปลี่ยนเป็น "extract", LEVELS_B ได้
items  = DATA[task]["dev"][:3]
scorer = score_extract if task == "extract" else score_classify
gold   = lambda it: it["label"] if "label" in it else f'{it["product"]} / {it["price"]} บาท'

print("=" * 72)
print("นี่ไม่ใช่ผลลัพธ์ — เป็น prompt ให้คัดลอกไปวางใน ChatGPT และ Claude")
print("แต่ละกรอบ = 1 ข้อความที่ต้องวางทั้งก้อน · Typhoon2 ตอบไว้ให้แล้วเพื่อเทียบ")
print("=" * 72)

sheet = []
n = 0
for lvl in ["L0 zero-shot", "L3 + บทบาท"]:
    system, fn = levels[lvl]
    for it in items:
        n += 1
        out, _ = ask(system, fn(it["text"]))
        sc = scorer(out, it)
        sheet.append({"#": n, "ระดับ": lvl, "เฉลย": gold(it), "Typhoon2 ตอบ": out[:40], "Typhoon2 คะแนน": sc,
                      "ChatGPT คะแนน": "", "Claude คะแนน": ""})
        print(f"\n━━━ prompt #{n} · {lvl} · เฉลย: {gold(it)} ━━━ (คัดลอกเฉพาะข้อความระหว่างเส้นประ)")
        print("-" * 60)
        if system: print(system)              # หน้าเว็บไม่มีช่อง system → วางบทบาทเป็นบรรทัดแรกของข้อความเดียวกัน
        print(fn(it["text"]))
        print("-" * 60)
        print(f"→ Typhoon2 ตอบ: {out[:60]!r}   คะแนน {sc} ({'ถูก' if sc == 1 else 'ผิด/ไม่ครบ'})")

print("\n\nตารางบันทึกผล (เติมสองคอลัมน์สุดท้ายเอง: 1 = ถูก, 0 = ผิด; โจทย์ (ข) ให้คะแนน 0–1 ตามเกณฑ์ 3.1):")
pd.DataFrame(sheet)

**อ่านผลอย่างไร** — เทียบ 3 คอลัมน์ "คะแนน" ในแต่ละระดับ

- ถ้า ChatGPT/Claude ตอบถูกตั้งแต่ L0 แต่ Typhoon2 ต้อง L3 ถึงจะถูก → โมเดลใหญ่ต้องการ prompt น้อยกว่า (มีความรู้และ instruction tuning มากกว่า)
- ถ้า L0 ของโมเดลใหญ่ตอบ **ยาวเป็นย่อหน้า** แทนที่จะตอบชื่อหมวด → ถือว่า "ผิดรูปแบบ" แม้เนื้อหาถูก — นี่คือหลักฐานว่าหลักการ "ระบุรูปแบบ output" ยังมีผลกับทุกโมเดล
- ถ้าทุกโมเดลถูกหมดทั้งสองระดับ → โจทย์นี้ง่ายเกินไปสำหรับเทียบ ลองเปลี่ยนเป็น `"extract", LEVELS_B`

## 3.5 บันทึก prompt ที่ดีที่สุดสำหรับ Lab 4
เลือกระดับที่คะแนน dev สูงสุด **จาก L1–L3** ของแต่ละโจทย์ (Lab 4 ต้องการ "prompt ที่ออกแบบแล้ว" มาเทียบกับ fine-tune · L0 เป็นเส้นฐานอยู่แล้ว) · คะแนนเสมอกัน → เลือกระดับที่สั้นกว่า
ดาวน์โหลด `best_prompts.json` จากแผง Files (ไอคอนโฟลเดอร์ซ้ายมือ) ไปอัปโหลดใน Lab 4

In [ ]:
best = {}
for task, levels in [("extract", LEVELS_B), ("classify", LEVELS_C)]:
    cand = df[(df["โจทย์"] == task) & (df["ระดับ prompt"] != "L0 zero-shot")]          # เลือกจาก L1–L3
    sub = cand.sort_values("คะแนน dev", ascending=False, kind="stable").iloc[0]            # เสมอกัน → ระดับที่สั้นกว่า
    name = sub["ระดับ prompt"]; system, fn = levels[name]
    best[task] = {"level": name, "system": system, "user_template": fn("{TEXT}"), "dev_score": float(sub["คะแนน dev"])}
    print(task, "→", name, sub["คะแนน dev"])
json.dump(best, open("best_prompts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(open("best_prompts.json", encoding="utf-8").read()[:600])

## ✅ สรุป Lab 3
- วัดผลด้วยชุด dev และเปลี่ยน prompt ทีละตัวแปร → รู้ว่าอะไรช่วยจริง
- โจทย์ที่ prompt เต็ม (L3) แล้วยังไม่ถึงเป้า (dev < 0.9) คือตัวเลือกสำหรับ fine-tune ใน Lab 4
- **ดาวน์โหลด `best_prompts.json`** จากแผง Files ก่อนปิด runtime (ไฟล์ใน Colab หายเมื่อปิด)
- **ห้ามเปิดชุด test** จนกว่าจะถึงขั้นสุดท้ายของ Lab 4